# Import libraries

In [98]:
import pandas as pd
from datetime import date

import dash
from dash import html, dcc, Output, Input
import dash_bootstrap_components as dbc
import plotly.graph_objects as go
import plotly.express as px

# Main parameters

In [99]:
LOGO = 'https://cdn.cookielaw.org/logos/b51d2c9b-8d53-4695-b601-f32cb832cef7/56478ccc-982c-427f-bdfe-b87cb415ba0f/d9ced935-47fc-4122-a1c2-a45bf306d6f9/spencerstuart_blue_trans.png'    # LOGO

PRIMARY = 'rgb(22, 48, 86)'                                     # primary color
SECONDARY = 'rgb(251, 204, 17)'                                 # secondary color
FONT = 'sans-serif'                                              # font type: also Arial, sans-serif

THEME = dbc.themes.SANDSTONE
TODAY = date.today()

# Upload the data

In [100]:
engagements_init = pd.read_csv('data/engagements.csv')
milestones_init = pd.read_csv('data/milestones.csv')

In [101]:
# fix the column datatype
engagements_init['start_date'] = pd.to_datetime(engagements_init['start_date'], format='%Y-%m-%d')
engagements_init['close_date'] = pd.to_datetime(engagements_init['close_date'], format='%Y-%m-%d')
milestones_init['milestone_date'] = pd.to_datetime(milestones_init['milestone_date'], format='%Y-%m-%d')
MAX_DATE = max(engagements_init['start_date'].max(), engagements_init['close_date'].max())
# filter todo: add filtration and box bplot information about errors/warnings

In [102]:
# filtration
wrong_closed = engagements_init[
    (engagements_init['status'] == 'Open') & ~(engagements_init['close_date'].isnull())
]['engagement_id']
missed_closed_data = engagements_init[
    (engagements_init['status'] == 'Closed') & (engagements_init['close_date'].isnull())
]['engagement_id']
missed_lead = engagements_init[(engagements_init['lead_partner'].isnull())]['engagement_id']

# drop the wrong closed engagements
engagements = engagements_init[~(engagements_init['engagement_id'].isin(wrong_closed))]
milestones = milestones_init[~(milestones_init['engagement_id'].isin(wrong_closed))]

# App elements

## Navigation bar

In [103]:
navbar = dbc.Row(dbc.Col(dbc.Navbar([
    dbc.Row([
        dbc.Col(html.Img(id='logo', src=LOGO, style={'height': '50px'}),),
        dbc.Col(
            dbc.NavbarBrand(id='Title', children='Automated Reporting',
                            style={'font-size': '32px', 'font-family': 'Georgia, serif',
                                   'margin-left': '20px', 'margin-right': 'auto'},
                            ), align='center'
        ),
    ]),
    dbc.RadioItems(id='select-time',
                   class_name='btn_group',
                   input_class_name='btn_check',
                   label_class_name='btn btn-outline-light',
                   label_checked_class_name='active',
                   options=[
                       {'label': 'month', 'value': 'month'},
                       {'label': 'quarter', 'value': 'quarter'},
                       {'label': 'year', 'value': 'year'},
                   ],
                   value='month',
                   style={'margin-left': 'auto', 'margin-right': '20px'})
])))

## Filtration

In [104]:
region_options = [{'label': region, 'value': region} for region in engagements['region'].unique()]  # todo: do if accurate
region_options.append({'label': 'all', 'value': 'all'})

practice_options = [{'label': practice, 'value': practice} for practice in engagements['practice'].unique()]
practice_options.append({'label': 'all', 'value': 'all'})

In [105]:
filter_card = dbc.CardBody(dbc.Row([
    dbc.Col([
        html.H6('Select a region'),
        dcc.Dropdown(id='dropdown-region', options=region_options, value='all', placeholder='Select a region'),
    ]),
    dbc.Col([
        html.H6('Select an practice'),  # todo: 'practice' or industry?
        dcc.Dropdown(id='dropdown-practice', options=practice_options, value='all', placeholder='Select an industry'),
    ])
]))

## Information Cards

In [106]:
# filters
# total revenue
# opened engagements
# closed engagements
# cancelled engagements
# average closed time
# acceptance rate (percentage of closed engagements)  # todo: how to calculate?

In [107]:
# creating card content
def get_info_cards(eng_df, min_date, prev_min_date):
    def get_content(title, amount, diff, flag_valute=False):
        def get_sign():
            return '-' if diff < 0 else '+'
        if flag_valute:
            main_value = f'{round(amount)} kEUR'
            comparison_value = f'<sub>{get_sign()} {abs(round(diff))} kEUR</sub>'
        else:
            main_value = str(amount)
            comparison_value = f'<sub>{get_sign()}{abs(diff)}</sub>'

        content = dbc.CardBody([
            html.H6(title, style={'fontWeight': 'lighter', 'textAlign': 'center'}),
            html.H3(main_value, style={'textAlign': 'center'}),
            dcc.Markdown(dangerously_allow_html=True,
                         children=comparison_value,
                         style={'textAlign': 'center'})
        ])
        return content

    # filtration
    filtered_ids = eng_df['engagement_id'].unique()

    df_closed_cur = eng_df[eng_df['close_date'] >= min_date]
    df_closed_prev = eng_df[(eng_df['close_date'] < min_date) & (eng_df['close_date'] >= prev_min_date)]

    # calculate only for CLOSED engagements  # todo: check for opened
    # total revenue
    revenue_cur = df_closed_cur['fee_kEUR'].sum()
    revenue_prev = df_closed_prev['fee_kEUR'].sum()

    # number of opened engagements
    opened_cur = eng_df[eng_df['start_date'] >= min_date].shape[0]
    opened_prev = eng_df[
        (eng_df['start_date'] < min_date) & (eng_df['start_date'] >= prev_min_date)
    ].shape[0]
    # number of closed engagements
    closed_cur = df_closed_cur[df_closed_cur['status'] == 'Closed'].shape[0]
    closed_prev = df_closed_prev[df_closed_prev['status'] == 'Closed'].shape[0]
    # number of cancelled engagements
    cancelled_cur = df_closed_cur[df_closed_cur['status'] == 'Cancelled'].shape[0]
    cancelled_prev = df_closed_prev[df_closed_prev['status'] == 'Cancelled'].shape[0]

    # average closing time
    close_time_cur = (
            df_closed_cur[df_closed_cur['status'] == 'Closed']['close_date'] -
            df_closed_cur[df_closed_cur['status'] == 'Closed']['start_date']
    ).mean()
    close_time_prev = (
            df_closed_prev[df_closed_prev['status'] == 'Closed']['close_date'] -
            df_closed_prev[df_closed_prev['status'] == 'Closed']['start_date']
    ).mean()

    # rate: closed / (closed + cancelled)
    cur_rate = closed_cur / (closed_cur + cancelled_cur) if closed_cur + cancelled_cur > 0 else 0
    prev_rate = closed_prev / (closed_prev + cancelled_prev) if closed_prev + cancelled_prev > 0 else 0

    return (
        get_content('Revenue', round(revenue_cur), revenue_cur - revenue_prev, True),
        get_content('Opened engagements', opened_cur, opened_cur - opened_prev),
        get_content('Closed engagements', closed_cur, closed_cur - closed_prev),
        get_content('Cancelled engagements', cancelled_cur, cancelled_cur - cancelled_prev),
        get_content('Average time (days)', close_time_cur.days, (close_time_cur - close_time_prev).days),
        get_content('Completed rate', round(cur_rate, 2), round(cur_rate - prev_rate, 2)),
    )

In [108]:
info_cards = [
    dbc.Col(dbc.Card(filter_card), style={'height': '100px'}),
    dbc.Col(dbc.Card(id='revenue-card'), style={'height': '100px'}),
    dbc.Col(dbc.Card(id='eng-opened-card'), style={'height': '100px'}),
    dbc.Col(dbc.Card(id='eng-closed-card'), style={'height': '100px'}),
    dbc.Col(dbc.Card(id='eng-cancelled-card'), style={'height': '100px'}),
    dbc.Col(dbc.Card(id='avg-time-card'), style={'height': '100px'}),
    dbc.Col(dbc.Card(id='acc-rate-card'), style={'height': '100px'}),
]

## Charts

In [109]:
margin = dict(l=30, r=5, t=60, b=20)
height, width = '250px', '500px'

In [110]:
chart_line_1 = [
    dbc.Col(dbc.Card(id='chart 1', style={'height': '350px'})),
    dbc.Col(dbc.Card(id='chart 2', style={'height': '350px'})),
    dbc.Col(dbc.Card(id='chart 3', style={'height': '350px'})),
]
chart_line_2 = [
    dbc.Col(dbc.Card(id='chart 4', style={'height': '350px'})),
    dbc.Col(dbc.Card(id='chart 5', style={'height': '350px'})),
]
chart_line_3 = [
    dbc.Col(dbc.Card(id='chart 6', style={'height': '350px'})),
    dbc.Col(dbc.Card(id='chart 7', style={'height': '350px'})),
    dbc.Col(dbc.Card(id='chart 8', style={'height': '350px'})),
    dbc.Col(dbc.Card(id='chart 9', style={'height': '350px'})),
]

In [111]:
# revenue by country
def get_rev_by_country(eng_df, min_date, prev_min_date):
    df_cur = eng_df[eng_df['close_date'] >= min_date][['country', 'fee_kEUR']].groupby('country').sum()
    y_cur, x_cur = df_cur['fee_kEUR'].values, df_cur['fee_kEUR'].index

    df_prev = eng_df[
        (eng_df['close_date'] < min_date) & (eng_df['close_date'] >= prev_min_date)
    ][
        ['country', 'fee_kEUR']
    ].groupby('country').sum()
    y_prev, x_prev = df_prev['fee_kEUR'].values, df_prev['fee_kEUR'].index

    fig = go.Figure(data=[
        go.Bar(
            y=y_cur, x=x_cur,
            name='current revenue',
        ),
        go.Bar(
            y=y_prev, x=x_prev,
            name='previous revenue',
        )
    ])
    fig.update_layout(
        title='Revenue by country',
        barmode='group',
        template='plotly_white',
        margin=margin,
        yaxis_title='kEUR',
    )
    return dbc.CardBody(
        dcc.Graph(figure=fig,
                  style={'height': height, 'width': width}
                  ),
    )


def get_conversion_chart(eng_df, min_date):  # IT LOOKS STRANGE, THINK ABOUT CHANGING IN THE FUTURE
    # calculate for closed deals
    # TODO: *compare to previous time period
    ids = eng_df[eng_df['close_date'] >= min_date]['engagement_id']
    df_cur = milestones[milestones['engagement_id'].isin(ids)][['milestone', 'candidate_count']].groupby('milestone').sum()
    data = pd.DataFrame({
        'label': label,
        'count': df_cur['candidate_count'][label],
    } for label in df_cur['candidate_count'].index)
    if len(data) == 0:
        return dbc.Col(
        dbc.Card(
            children='No data', style={'height': height}
        )
    )

    data['percentage'] = data['count'].apply(lambda x: round(x / df_cur['candidate_count']['Kickoff'] * 100))

    fig = px.funnel(data.sort_values('count', ascending=False), x='label', y='count')
    fig.update_layout(
        title='Conversion rate',
        template='plotly_white',
        margin=margin,
        showlegend=False,
    )
    fig.update_traces(
        texttemplate='%{customdata}%',  # show values with percentage sign
        textposition='inside',
        customdata=data.sort_values('count', ascending=False)['percentage'],  # attach percentages
        # marker_color=PRIMARY
    )
    return dbc.Col(
        dbc.Card(
            dcc.Graph(figure=fig), style={'height': height}
        )
    )


def get_pie_chart(eng_df, min_date):
    # todo: this is not the best metrics, because it's vain
    # pie chart with percentage for statuses for ALL NON-CLOSED engagements
    df_selected = eng_df[(eng_df['close_date'].isna()) & (eng_df['status'] != 'Closed')]['status'].value_counts()
    data = pd.DataFrame({
        'status': status,
        'count': count,
    } for status, count in zip(df_selected.index, df_selected.values))

    fig = px.pie(data, names='status', values='count')
    fig.update_layout(
        title='Status for non-closed engagements',
        template='plotly_white',
        margin=margin,
        yaxis=dict(
            title='',
            tickformat='.0%',
        ),
    )
    return dbc.Col(
        dbc.Card(
            dcc.Graph(figure=fig), style={'height': height}
        )
    )

In [112]:
def get_time_series(eng_df, min_date, timeframe):
    rev_cur = eng_df[eng_df['close_date'] >= min_date][['close_date', 'fee_kEUR']]
    rev_cur.set_index('close_date', inplace=True)
    if timeframe == 'month':
        rev_cur = rev_cur.resample('W')['fee_kEUR'].sum()
    else:
        rev_cur = rev_cur.resample('M')['fee_kEUR'].sum()

    x, y = rev_cur.index, rev_cur.values
    fig = go.Figure(data=go.Scatter(x=x, y=y, mode='lines+markers', name='current'),)
    fig.update_layout(
        title='Revenue',
        template='plotly_white',
        margin=margin
    )
    return dbc.Col(
        dbc.Card(
            dcc.Graph(figure=fig), style={'height': height}
        )
    )


def get_time_diff(eng_df, min_date):
    labels = ['Kickoff', 'Longlist', 'Shortlist', 'Offer', 'Accepted']
    ids = eng_df[eng_df['close_date'] >= min_date]['engagement_id']
    diff_data = []
    for eng_id in ids:
        diff_i = {}
        for j in range(1, len(labels)):
            end_time = milestones[
                (milestones['engagement_id'] == eng_id) & (milestones['milestone'] == labels[j])
            ]['milestone_date']
            start_time = milestones[
                (milestones['engagement_id'] == eng_id) & (milestones['milestone'] == labels[j - 1])
            ]['milestone_date']
            diff_i[f'{labels[j - 1]} -> {labels[j]}'] = (end_time.values[0] - start_time.values[0]).astype('timedelta64[D]')
        diff_data.append(diff_i)

    diff_df = pd.DataFrame(diff_data).mean().apply(lambda x: x.days)

    data = pd.DataFrame({'label': label, 'time': time} for label, time in zip(diff_df.index, diff_df.values))

    fig = px.bar(data, x='label', y='time')
    fig.update_layout(
        title='Average time',
        template='plotly_white',
        margin=margin,
        showlegend=False,
    )
    return dbc.Col(
        dbc.Card(
            dcc.Graph(figure=fig), style={'height': height}
        )
    )

In [113]:
def get_eng_by_level(eng_df, min_date, prev_min_date):
    df_cur = eng_df[eng_df['close_date'] >= min_date]['level'].value_counts()
    y_cur, x_cur = df_cur.values, df_cur.index

    df_prev = eng_df[
        (eng_df['close_date'] < min_date) & (eng_df['close_date'] >= prev_min_date)
    ]['level'].value_counts()
    y_prev, x_prev = df_prev.values, df_prev.index

    fig = go.Figure(data=[
        go.Bar(
            y=y_cur, x=x_cur,
            name='current',
        ),
        go.Bar(
            y=y_prev, x=x_prev,
            name='previous',
        )
    ])
    fig.update_layout(
        title='Level distribution',
        barmode='group',
        template='plotly_white',
        margin=margin,
        yaxis_title='engagement count',
    )
    return dbc.CardBody(
        dcc.Graph(figure=fig,
                  style={'height': height, 'width': width}
                  ),
    )


def get_eng_by_size(eng_df, min_date, prev_min_date):
    df_cur = eng_df[eng_df['close_date'] >= min_date]['client_size'].value_counts()
    y_cur, x_cur = df_cur.values, df_cur.index

    df_prev = eng_df[
        (eng_df['close_date'] < min_date) & (eng_df['close_date'] >= prev_min_date)
    ]['client_size'].value_counts()
    y_prev, x_prev = df_prev.values, df_prev.index

    fig = go.Figure(data=[
        go.Bar(
            y=y_cur, x=x_cur,
            name='current',
        ),
        go.Bar(
            y=y_prev, x=x_prev,
            name='previous',
        )
    ])
    fig.update_layout(
        title='Client size distribution',
        barmode='group',
        template='plotly_white',
        margin=margin,
        yaxis_title='engagement count',
    )
    return dbc.CardBody(
        dcc.Graph(figure=fig,
                  style={'height': height, 'width': width}
                  ),
    )


def get_eng_by_lead(eng_df, min_date, prev_min_date):
    df_cur = eng_df[eng_df['close_date'] >= min_date]['lead_partner'].value_counts()
    y_cur, x_cur = df_cur.values, df_cur.index

    df_prev = eng_df[
        (eng_df['close_date'] < min_date) & (eng_df['close_date'] >= prev_min_date)
    ]['lead_partner'].value_counts()
    y_prev, x_prev = df_prev.values, df_prev.index

    fig = go.Figure(data=[
        go.Bar(
            y=y_cur, x=x_cur,
            name='current',
        ),
        go.Bar(
            y=y_prev, x=x_prev,
            name='previous',
        )
    ])
    fig.update_layout(
        title='Closed engagements per lead partner',
        barmode='group',
        template='plotly_white',
        margin=margin,
        yaxis_title='engagement count',
    )
    return dbc.CardBody(
        dcc.Graph(figure=fig,
                  style={'height': height, 'width': width}
                  ),
    )

# Launch app

In [114]:
body_app = dbc.Container([
    dbc.Row(navbar),
    html.Br(),
    dbc.Row(info_cards),
    html.Br(),
    dbc.Row(chart_line_1),
    html.Br(),
    dbc.Row(chart_line_2),
    html.Br(),
    dbc.Row(chart_line_3),
    html.Br(),
], fluid=True)

In [115]:
app = dash.Dash(external_stylesheets=[THEME,])

app.layout = html.Div([body_app])

@app.callback(Output('revenue-card', 'children'),
              Output('eng-opened-card', 'children'),
              Output('eng-closed-card', 'children'),
              Output('eng-cancelled-card', 'children'),
              Output('avg-time-card', 'children'),
              Output('acc-rate-card', 'children'),
              Output('chart 1', 'children'),
              Output('chart 2', 'children'),
              Output('chart 3', 'children'),
              Output('chart 4', 'children'),
              Output('chart 5', 'children'),
              Output('chart 6', 'children'),
              Output('chart 7', 'children'),
              Output('chart 8', 'children'),
              [Input('select-time', 'value'),
               Input('dropdown-region', 'value'),
               Input('dropdown-practice', 'value'),])
def get_cards(timeframe, region, practice):
    # filtration
    region = 'all' if not region else region
    practice = 'all' if not practice else practice

    if region != 'all':
        eng_selected_1 = engagements[engagements['region'] == region]
    else:
        eng_selected_1 = engagements.copy()
    if practice != 'all':
        eng_selected = engagements[engagements['practice'] == practice]
    else:
        eng_selected = eng_selected_1.copy()

    if timeframe == 'quarter':
        min_date = MAX_DATE - pd.DateOffset(months=3)
        prev_min_date = MAX_DATE - pd.DateOffset(months=6)
    elif timeframe == 'month':
        min_date = MAX_DATE - pd.DateOffset(months=1)
        prev_min_date = MAX_DATE - pd.DateOffset(months=2)
    else:
        min_date = MAX_DATE - pd.DateOffset(years=1)
        prev_min_date = MAX_DATE - pd.DateOffset(years=2)

    # infocards
    (revenue_card,
     opened_card, closed_card, cancelled_card,
     time_card, rate_card) = get_info_cards(eng_selected, min_date, prev_min_date)
    # charts
    chart_1 = get_rev_by_country(eng_selected, min_date, prev_min_date)
    chart_2 = get_conversion_chart(eng_selected, min_date)
    chart_3 = get_pie_chart(eng_selected, min_date)
    chart_4 = get_time_series(eng_selected, min_date, timeframe)
    chart_5 = get_time_diff(eng_selected, min_date)
    chart_6 = get_eng_by_level(eng_selected, min_date, prev_min_date)
    chart_7 = get_eng_by_size(eng_selected, min_date, prev_min_date)
    chart_8 = get_eng_by_lead(eng_selected, min_date, prev_min_date)
    content = [revenue_card,
               opened_card, closed_card, cancelled_card,
               time_card, rate_card,
               chart_1, chart_2, chart_3,
               chart_4, chart_5,
               chart_6, chart_7, chart_8]
    return content


if __name__ == '__main__':
    app.run(debug=True)